# Tutorial 2: Model averaging for inclusion probabilities

In the final tutorial we apply the Bayesian model averaging procedure to data by Moiseenko et al. (in preparation). This study investigates the effects of social interaction and information gain in the environment, while children pay attention to toys appearing in boxes. Our outcome measure is the variable $y_{ist} \in \{0, 1\}$, which indicates whether an infant $i$ paid attention to the setup at sequence-of-toys $s$ and trial $t$.

![image](drawing.png)

As soon as an infant looks away ($y_{ist} =1$) the sequence stops and a new sequence starts. This means our data always looks like a series of zeros, followed by a single one (the looking-away event). Situations like this are well-described by so-called [Cox proportional hazards models](https://en.wikipedia.org/wiki/Proportional_hazards_model). In Bayesian terms, it is defined as:

$$
\begin{align*}
    H^0_t &\sim \text{Gamma}(0.1, 0.1) && t=1, \ldots, T\\
    \beta_j &\sim \mathcal{N}(0.0, 1.0) && j=1, \ldots, D \\
    \lambda_{ist} &= H^0_t \exp\left(\bm{\beta} X\right) &&\\
    y_{ist} &\sim \text{Poisson}(\lambda_{ist}) &&
\end{align*}
$$
where $i=1,\ldots, N, s=1, \ldots, S, t=1, \ldots, T$ indicate the repetitions over infants, sequences, and trials. $\lambda_{ist}$ is the hazard rate and is the key quantity of the model. It reflects the conditional probability of looking away, given that you have not looked away yet (the trials are certainly _not_ independent). It is constructed by combinining several terms. The first of those is $H^0_t$, the _baseline hazard rate_, indicating how likely children are to look away at every trial in a sequence of toys appearing.

The next is the combination of coefficients $\beta$ and predictors $x$. The matrix $X$ is essentially a design matrix and contains the values for our predictors: the sequence number, whether the infant's caregiver was free to interact with them, and the information gain from the stimulus (derived from how predictable the location of the appearing toy was). The coefficients $\beta$ indicate the relative importance of each of these predictors.  However, we are not only interested in whether these predictors affect the probability of infants looking away, but also whether there are effects from their interactions. 

Crucial for our purposes today is that we do not know which predictors and interactions are truly relevant. If we put one of them in, we implicitly say we _do_ think this term is relevant - so just using everything is not the right idea. Can we do better?

Here we construct a **Bayesian model averaged analysis of the Cox proportional hazards model**. There are a total of $M=19$ different potential models in this scenario, so we have to pay attention to how we do this!

In [ ]:
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)

import jax.random as jrnd
import jax.numpy as jnp
from jax.scipy.special import logsumexp

import blackjax
import numpyro as npr
import numpyro.distributions as dist
import numpyro.distributions.transforms as nprb
import time
import sys
import os

import bamojax
from bamojax.base import Model
from bamojax.samplers import mcmc_sampler, gibbs_sampler
from bamojax.inference import MCMCInference, SMCInference
from bamojax.marginal_likelihoods.bridge_sampling import bridge_sampling
from bamojax.marginal_likelihoods.thames import thames
from bamojax.marginal_likelihoods.laplace import laplace_approximation
from bamojax.marginal_likelihoods.naive_monte_carlo import naive_monte_carlo
from bamojax.marginal_likelihoods.importance_sampling import importance_sampling

print('Python version:     ', sys.version)
print('Jax version:        ', jax.__version__)
print('BlackJax version:   ', blackjax.__version__)
print('Bamojax version:    ', bamojax.__version__)
print('Numpyro version:    ', npr.__version__)
print('Jax default backend:', jax.default_backend())
print('Jax devices:        ', jax.devices())

from tutorial2_utility import load_infant_data, interaction_models, index2name, build_design_matrix, ix2term

### plotting

plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

SMALL_SIZE = 12
MEDIUM_SIZE = 16
BIGGER_SIZE = 18

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=BIGGER_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=MEDIUM_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

def pretty_axis(ax, fix_xaxis=True, fix_yaxis=True):
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False) 
    if fix_xaxis:
        ax.spines['bottom'].set_position(('axes', -0.05))
    if fix_yaxis:
        ax.spines['left'].set_position(('axes', -0.05))

#

Load the data (the `exposure matrix` applies a mask; only observations up to and including the looking-away event are included in computing the likelihood).

The function `index2name` you might use later on as well: it takes the list of models and an index, and returns a string showing which predictors are included in the given model.

In [ ]:
X, Y, exposure_matrix, varnames = load_infant_data()
N, S, T, D = X.shape

models = interaction_models([0, 1, 2])
M = len(models)
for i, m in enumerate(models):
    print(f'Model index: {i}, contents: {index2name(models, i)}')

all_design_matrices = [build_design_matrix(X, model) for model in models] 

Define the model(s) using `bamojax` notation:

In [4]:
def setup_cph_model(X, Y, exposure_matrix, name='CPH'):

    def cph_link_fn(h0, beta, E, X):
        """ The Cox proportional hazards link function.
        
        """
        mu = jnp.tensordot(beta, X, axes=[0, 3])
        lambda_ = E * h0 * jnp.exp(mu) + 1e-20  
        return dict(rate=lambda_)

    #
    _, _, T, D = X.shape

    cph_model = Model(name)
    h0_node = cph_model.add_node('h0', distribution=dist.Gamma(concentration=0.1, rate=0.1), shape=(T, ))
    beta_node = cph_model.add_node('beta', distribution=dist.Normal(loc=0.0, scale=1.0), shape=(D, ))
    input_node = cph_model.add_node('X', observations=X)
    exposure_node = cph_model.add_node('E', observations=exposure_matrix)
    obs_node = cph_model.add_node('Y', distribution=dist.Poisson, observations=Y, 
                                parents=dict(h0=h0_node, 
                                            beta=beta_node, 
                                            E=exposure_node, 
                                            X=input_node), 
                                link_fn=cph_link_fn)
    return cph_model

#
def setup_cph_null_model(Y, exposure_matrix, name):
    """The null model we define slightly differently.
    """

    def cph_link_fn(h0, E):
        """ The Cox proportional hazards link function.
        
        """
        lambda_ = E * h0 * jnp.exp(0) + 1e-20  
        return dict(rate=lambda_)

    #
    _, _, T = Y.shape

    cph_model = Model(name)
    h0_node = cph_model.add_node('h0', distribution=dist.Gamma(concentration=0.1, rate=0.1), shape=(T, ))
    exposure_node = cph_model.add_node('E', observations=exposure_matrix)
    obs_node = cph_model.add_node('Y', distribution=dist.Poisson, observations=Y, 
                                parents=dict(h0=h0_node, 
                                            E=exposure_node),
                                link_fn=cph_link_fn)
    return cph_model

#

cph_models = []
cph_models.append(setup_cph_null_model(Y, exposure_matrix, 'CPH_Null'))

for m in range(1, M):
    cph_models.append(setup_cph_model(all_design_matrices[m], Y, exposure_matrix, f'CPH_{m}'))

We now want to perform a Bayesian model averaging analysis over these 19 models. Your goal is to:

- Estimate the log marginal likelihood of each model.
- Compute the posterior model probability of each model.
- Compute, for each predictor / interaction effect, the probability of it being present under the
  - prior model probabilities ($p(\mathcal{M}_i)=1/M=1/19$),
  - posterior model probabilities.
- Compute the inclusion Bayes factor per predictor / interaction effect.

Computing the posterior samples for all these models is unfortunately a time-intensive process. This is why I already have collected 4 chains of 2000 samples each for every model, using NUTS HMC. The following code block loads these and merges the chains, so that there are 8000 samples for each model.

In [ ]:
def get_posterior(suffix=''):
    filename = f'samples/samples_CPH_{suffix}.npz'    
    if os.path.exists(filename):
        posterior_samples = jnp.load(filename, allow_pickle=True)['samples'].item()        
        # merge the 4 independent chains
        posterior_samples_merged = jax.tree.map(lambda x: jnp.reshape(x, shape=(x.shape[0]*x.shape[1], -1)), posterior_samples)
    return posterior_samples_merged

#

posterior_samples = []

for m in range(M):
    cph_model = cph_models[m]
    print(f'Loading samples for model {index2name(models, m)}')
    posterior_samples.append(get_posterior(suffix=m))

### Queston 1 - the marginal likelihoods

You can again use different techniques for this. As the models become more complex (the 'full' model has 7 parameters, several of which will be correlated to some degree), the baselines like Naive Monte Carlo will most likely fail. 

Depending on which method you choose, you may have to come up with additional decisions:
- Importance sampling: Define an importance distribution.
- Bridge sampling: Define bijectors - for this implementationthese apply transformations _from_ continuous numbers _to_ the desired range. For example, if you have a variable in your model $\theta\in [0,1]$, you could use a `ProbitBijector`. Bijectors are available under the `nprb` module.
- Sequential Monte Carlo: Define an MCMC kernel for the particle mutations.

Compute the log marginal likelihood of each of the 19 models.

_Note: although the log marginal likelihood estimation procedures are faster than estimating the posterior, this might still take a few minutes. Try estimating $\log p(\mathcal{D}\mid\mathcal{M})$ for one model first, before applying your code to all of them._

### Question 2 - the posterior model probabilities

Now convert the log marginal likelihoods to posterior model probabilities. You will need the `logsumexp` trick like in Tutorial 1.

Which model is most likely?

### Question 3 - the impact of different models

Now we are ready to plot the parameter estimates _for each different model_. Since there are 19 models, that is quite some plots. For now, we'll focus only on the coefficients $\beta_{\text{IG*Cond}}$, which represents the _interaction effect_ between the stimulus information gain, and the social/nonsocial condition that the infant was in.

Plot, for all models that contain this predictor, the parameter distribution of $\beta_{\text{IG*Cond}}$. You can do this with a histogram, or if you want it to look fancy, use a [kernel density estimate](https://docs.jax.dev/en/latest/_autosummary/jax.scipy.stats.gaussian_kde.html). Try plotting in the same figure, but with different colors/transparancy, and add a legend indicating to which model those samples corresponded. In the legend, also include the posterior model probabilities of the models.


### Question 4 - The effect of the model probabilities

Now here is a crucial observation: We see that different models assign different densities to this predictor. Some are roughly similar, but one is decidedly different. This model has a large chunk of the posterior probability, but at the same time there is plenty of uncertainty remaining!

If you sum up the model probabilities of the models you visualized here (in terms of the slides, you showed $M : \gamma_{\text{IG*Cond}}=1$, that is, those models that included this predictor), what do you get? What does that mean for $\beta_{\text{IG*Cond}}$ under the remaining model mass?

### Question 5 - The BMA of $\beta_{\text{IG*Cond}}$

Visualize the distribution 

$$
    p(\beta_{\text{IG*Cond}} \mid \mathcal{D}) = \sum_{i=1}^M p(\beta_{\text{IG*Cond}} \mid \mathcal{D}, M_i) p(M_i \mid \mathcal{D}) \enspace,
$$
which is the Bayesian model average distribution of this predictor.

Just like in tutorial 1, you can obtain a sampled approximation of this distribution by taking $[S_i]= S \cdot p(\mathcal{M}\mid \mathcal{D})$ samples from each distribution ($[x]$ rounds $x$ to the nearest integer) and stacking these all together. In case the model does not contain $\beta_{\text{IG*Cond}}$, all of its samples return zero so you need to add the appropriate amount of zero samples!

Plot the histogram in the same plot as your previous figure, so we can see the difference between the estimates by each model, and the BMA.

The figure shows the power of Bayesian model averaging: the estimate is not a simple unimodal distribution, but a mixture of multiple distributions, in this case containing three peaks: one belonging to the models that do not contain the predictor at all, one from the models that do contain it, but do not have the triple interaction $\beta_{\text{IG*Seq*Cond}}$, and finally the one corresponding to the model containing everything.

Note that of course we can repeat the analysis above for every predictor; I picked this one as it clearly demonstrates the effect.

### Question 6 - Posterior model probabilities, Bayes factors, and inclusion probabilities

Sort the models by their posterior probability, and then print a table (just print statements line-by-line if you please), showing four columns: 

1. The name of the model (which predictors are in),
2. The prior model probability,
3. The posterior model probability, and
4. The log Bayes factor comparing the top model to the i-th model (this means the first row should have entry 0 here, as that would compare the model with itself).


This tells us a part of the story: It shows the most likely model, but also shows that there are a couple of models that are not outperformed that strongly by the top model (see the [Bayes factor interpretation table](https://en.wikipedia.org/wiki/Bayes_factor#Interpretation) to get a bit of a feel for Bayes factors if you don't have that yet - note that we are looking at the _logarithm_ of Bayes factors here!).

However, to know which predictors are important, we need their _inclusion probabilities_: in how many models are they present, weighted by the posterior model probabilities?

Print another table with 4 columns:

1. The name of the predictor (main effect, interaction, or three-way interaction).
2. The prior inclusion probability of this term (how many models contain this predictor, weighted by their _prior_ probabilities),
3. The posterior inclusion probability of this term (how many models contain this predictor, weighted by their _posterior_ probabilities), and
4. The inclusion log Bayes factor (see slides); the log of the summed marginal likelihoods of all models _with_ the predictor, divided by the summed marginal likelihoods of models without the predictor. Note that here you will need the `logsumexp` trick again.

Consider what this implies: even though the model that includes the three-way interaction term is by itself the most likely model, it still accounts for 'only' about 0.52 probability. That means with 0.48 probability, the predictor should be included - which is roughly 50/50, suggesting a Bayes factor of 1 (and so a log Bayes factor of close to zero)!

Other predictors, like 'Seq' were already a priori in many models, but after seeing data we are completely convinced this term is relevant (it has a massive Bayes factor, try computing $e^{138.57}$). 

The key observation is that _selecting_ the most likely model gives a very different conclusion than _averaging_ over model uncertainty. In the former, we are claiming the three-way interaction effect is there, in the latter we see that its basically a coin toss - we don't really find evidence either way. Imagine the potentially different titles in the corresponding Nature paper! 

This is way Bayesian model averaging can be crucial: we often take for granted that we have picked the right model, but in reality several alternatives might exist.